# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [2]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

c:\Users\david\AppData\Local\Programs\Python\Python313\Lib\site-packages\beir\util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets\scifact.zip: 100%|██████████| 2.69M/2.69M [00:44<00:00, 62.9kiB/s]


'../data/beir_datasets\\scifact'

In [4]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

100%|██████████| 5183/5183 [00:00<00:00, 116803.29it/s]


In [5]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [6]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [7]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [8]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

Para el retrieval inicial se usó BM25 como baseline. La idea es tener un primer ranking
contra el cual comparar los re-rankers de las partes siguientes.

Se tokenizó el corpus concatenando título y texto de cada documento, y se aplicó una
tokenización simple (lowercase + split por espacios). No se usó stemming ni remoción de
stopwords porque los modelos de re-ranking posteriores trabajan con texto crudo, y se
quería mantener consistencia en el preprocesamiento.

Se usó la librería `rank_bm25` con los parámetros estándar (k1=1.5,
b=0.75). Para cada query se recuperaron los top-100 documentos en vez de solo 10, porque
ese pool de candidatos es el que luego van a reordenar el cross-encoder y el modelo LTR.
Si solo se recuperaran 10, los re-rankers no podrían rescatar documentos relevantes que
BM25 dejó en posiciones más bajas.

Se evaluó el baseline con tres métricas: nDCG@10, que a diferencia de Precision toma en
cuenta grados de relevancia (SciFact usa relevancia 0, 1 y 2); Recall@10, que mide qué
fracción de los documentos relevantes cayó en el top-10; y MAP@10, que premia que los
documentos relevantes aparezcan en las primeras posiciones del ranking.

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from collections import defaultdict
import math

# --- Tokenización simple para BM25 ---
def tokenize(text):
    return text.lower().split()

# Tokenizar corpus
corpus_ids = df_corpus["doc_id"].tolist()
corpus_texts = (df_corpus["title"].fillna("") + " " + df_corpus["text"].fillna("")).tolist()
corpus_tokenized = [tokenize(t) for t in corpus_texts]

bm25 = BM25Okapi(corpus_tokenized)

# --- Funciones de métricas ---
def dcg_at_k(scores, k):
    scores = np.array(scores[:k])
    return np.sum(scores / np.log2(np.arange(2, len(scores) + 2)))

def ndcg_at_k(retrieved_ids, qrel_dict, k):
    # Gains del ranking obtenido
    gains = [qrel_dict.get(doc_id, 0) for doc_id in retrieved_ids[:k]]
    dcg = dcg_at_k(gains, k)
    # Ideal: ordenar por relevancia descendente
    ideal_gains = sorted(qrel_dict.values(), reverse=True)[:k]
    idcg = dcg_at_k(ideal_gains, k)
    return dcg / idcg if idcg > 0 else 0.0

def recall_at_k(retrieved_ids, qrel_dict, k):
    relevant = {did for did, rel in qrel_dict.items() if rel > 0}
    if len(relevant) == 0:
        return 0.0
    retrieved_set = set(retrieved_ids[:k])
    return len(retrieved_set & relevant) / len(relevant)

def average_precision(retrieved_ids, qrel_dict):
    relevant = {did for did, rel in qrel_dict.items() if rel > 0}
    hits = 0
    sum_prec = 0.0
    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant:
            hits += 1
            sum_prec += hits / (i + 1)
    return sum_prec / len(relevant) if len(relevant) > 0 else 0.0

# --- Construir qrels como dict de dicts: {query_id: {doc_id: relevance}} ---
qrels_dict = defaultdict(dict)
for _, row in df_qrels.iterrows():
    qrels_dict[row["query_id"]][row["doc_id"]] = row["relevance"]

# --- BM25 retrieval para todas las queries ---
K = 100  # recuperar top-100 para luego re-rankear
TOP_EVAL = 10

bm25_results = {} 
bm25_scores_dict = {}

for _, qrow in df_queries.iterrows():
    qid = qrow["query_id"]
    query_tokens = tokenize(qrow["query"])
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:K]
    
    retrieved = [corpus_ids[i] for i in top_indices]
    bm25_results[qid] = retrieved
    bm25_scores_dict[qid] = {corpus_ids[i]: float(scores[i]) for i in top_indices}

# --- Evaluar BM25 baseline ---
bm25_ndcg, bm25_recall, bm25_ap = [], [], []

for qid in df_queries["query_id"]:
    if qid not in qrels_dict:
        continue
    retrieved = bm25_results[qid]
    qrel = qrels_dict[qid]
    
    bm25_ndcg.append(ndcg_at_k(retrieved, qrel, TOP_EVAL))
    bm25_recall.append(recall_at_k(retrieved, qrel, TOP_EVAL))
    bm25_ap.append(average_precision(retrieved[:TOP_EVAL], qrel))

print("=== BM25 Baseline ===")
print(f"nDCG@10:   {np.mean(bm25_ndcg):.4f}")
print(f"Recall@10: {np.mean(bm25_recall):.4f}")
print(f"MAP@10:    {np.mean(bm25_ap):.4f}")

=== BM25 Baseline ===
nDCG@10:   0.5597
Recall@10: 0.6862
MAP@10:    0.5147


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

El cross-encoder recibe la query y el documento concatenados
y los procesa juntos, lo que le permite capturar interacciones entre las palabras de ambos
que un modelo como BM25 no puede ver. BM25 compara términos de forma independiente,
mientras que el cross-encoder entiende el contexto completo del par query-documento.

El tradeoff es velocidad: BM25 puntúa miles de documentos en milisegundos, pero el
cross-encoder necesita una pasada completa del transformer por cada par. Por eso no se
corre sobre todo el corpus sino solo sobre los 100 candidatos que BM25 ya recuperó. Este
es el principio del pipeline de dos etapas: una primera etapa rápida filtra, y una segunda
etapa lenta pero precisa reordena.

Se usó el modelo `ms-marco-MiniLM-L-6-v2`, para predecir
relevancia de pares query-documento. Para optimizar tiempo, se armaron todos los pares
de todas las queries en una sola lista (300 queries × 100 candidatos = 30,000 pares) y se
evaluaron en un solo llamado con batch_size=256, en vez de llamar al modelo query por query.

Al final se compara el top-10 antes y después del re-ranking para una query de ejemplo,
mostrando qué documentos subieron, bajaron o entraron al top-10. Un documento que BM25
tenía en posición 30 pero el cross-encoder subió a posición 1 es un caso donde la
comprensión semántica del cross-encoder rescató un resultado que el matching por términos
no valoró.

In [ ]:
from sentence_transformers import CrossEncoder
from collections import defaultdict

# Modelo cross-encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Mapa rápido doc_id -> texto
id_to_text = dict(zip(df_corpus["doc_id"], corpus_texts))

all_pairs = []
pair_map = []

for _, qrow in df_queries.iterrows():
    qid = qrow["query_id"]
    query_text = qrow["query"]
    candidates = bm25_results[qid][:K]
    
    for doc_id in candidates:
        all_pairs.append((query_text, id_to_text[doc_id]))
        pair_map.append((qid, doc_id))

print(f"Total de pares a evaluar: {len(all_pairs)}")

all_scores = cross_encoder.predict(all_pairs, batch_size=256, show_progress_bar=True)

# --- Reconstruir rankings por query ---
ce_raw = defaultdict(list)
for (qid, doc_id), score in zip(pair_map, all_scores):
    ce_raw[qid].append((doc_id, float(score)))

ce_results = {}
for qid, pairs in ce_raw.items():
    pairs.sort(key=lambda x: x[1], reverse=True)
    ce_results[qid] = [doc_id for doc_id, _ in pairs]

# --- Mostrar cambios de posición para una query de ejemplo ---
example_qid = "133"
print(f"\nQuery: {df_queries.loc[df_queries['query_id'] == example_qid, 'query'].values[0]}\n")

print(f"{'Doc ID':<12} {'BM25 Rank':>10} {'CE Rank':>10} {'Cambio':>8}")
print("-" * 45)

bm25_top10 = set(bm25_results[example_qid][:10])
ce_top10 = set(ce_results[example_qid][:10])
all_top10 = bm25_top10 | ce_top10

bm25_rank_map = {did: i+1 for i, did in enumerate(bm25_results[example_qid][:K])}
ce_rank_map = {did: i+1 for i, did in enumerate(ce_results[example_qid][:K])}

for doc_id in sorted(all_top10, key=lambda d: ce_rank_map.get(d, 999)):
    br = bm25_rank_map.get(doc_id, "-")
    cr = ce_rank_map.get(doc_id, "-")
    if isinstance(br, int) and isinstance(cr, int):
        delta = br - cr
        symbol = f"+{delta}" if delta > 0 else str(delta)
    else:
        symbol = "new" if br == "-" else "out"
    print(f"{doc_id:<12} {str(br):>10} {str(cr):>10} {symbol:>8}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 14576.09it/s]


Total de pares a evaluar: 30000


Batches: 100%|██████████| 118/118 [17:17<00:00,  8.79s/it]


Query: Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Doc ID        BM25 Rank    CE Rank   Cambio
---------------------------------------------
35660758             30          1      +29
12640810              6          2       +4
16280642             37          3      +34
36345185             78          4      +74
6969753              10          5       +5
9507605               2          6       -4
86694016              8          7       +1
19752008             20          8      +12
17934082              9          9        0
21295300             71         10      +61
37964706              3         17      -14
12785130              5         72      -67
26688294              1         81      -80
30861948              7         84      -77
5270265               4         90      -86


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [13]:
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import GradientBoostingRegressor

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

# --- Pre-computar embeddings (una sola vez cada uno) ---
print("Generando embeddings del corpus...")
corpus_embeddings = bi_encoder.encode(corpus_texts, show_progress_bar=True, batch_size=64)
corpus_norms = np.linalg.norm(corpus_embeddings, axis=1, keepdims=True) + 1e-9
corpus_embeddings_norm = corpus_embeddings / corpus_norms

print("Generando embeddings de queries...")
query_texts_list = df_queries["query"].tolist()
query_ids_list = df_queries["query_id"].tolist()
query_embeddings = bi_encoder.encode(query_texts_list, show_progress_bar=True, batch_size=64)
query_norms = np.linalg.norm(query_embeddings, axis=1, keepdims=True) + 1e-9
query_embeddings_norm = query_embeddings / query_norms

# Mapas de acceso rápido
id_to_idx = {did: i for i, did in enumerate(corpus_ids)}
qid_to_idx = {qid: i for i, qid in enumerate(query_ids_list)}

# --- Construir features vectorizado ---
def build_features_batch(qid, query_text, candidates, bm25_scores):
    """Genera matriz de features para todos los candidatos de una query de golpe."""
    q_idx = qid_to_idx[qid]
    q_emb = query_embeddings_norm[q_idx]  # ya precomputado
    q_tokens = set(tokenize(query_text))
    
    feats = []
    for doc_id in candidates:
        d_idx = id_to_idx[doc_id]
        
        # F1: BM25 score
        f_bm25 = bm25_scores.get(doc_id, 0.0)
        
        # F2: Coseno (dot product de vectores ya normalizados)
        cos_sim = float(np.dot(q_emb, corpus_embeddings_norm[d_idx]))
        
        # F3: Longitud del documento
        f_doc_len = len(corpus_texts[d_idx].split())
        
        # F4: Term overlap
        d_tokens = set(tokenize(corpus_texts[d_idx]))
        overlap = len(q_tokens & d_tokens) / len(q_tokens) if q_tokens else 0
        
        feats.append([f_bm25, cos_sim, f_doc_len, overlap])
    
    return np.array(feats)

# --- Dataset de entrenamiento ---
print("Construyendo features de entrenamiento...")
X_train, y_train = [], []

for qid in list(qrels_dict.keys()):
    if qid not in bm25_results:
        continue
    query_text = df_queries.loc[df_queries["query_id"] == qid, "query"].values
    if len(query_text) == 0:
        continue
    query_text = query_text[0]
    
    candidates = bm25_results[qid][:K]
    qrel = qrels_dict[qid]
    
    feats = build_features_batch(qid, query_text, candidates, bm25_scores_dict[qid])
    labels = [qrel.get(doc_id, 0) for doc_id in candidates]
    
    X_train.append(feats)
    y_train.extend(labels)

X_train = np.vstack(X_train)
y_train = np.array(y_train)
print(f"Dataset: {X_train.shape[0]} pares, {X_train.shape[1]} features")

# --- Entrenar LTR ---
ltr_model = GradientBoostingRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42
)
ltr_model.fit(X_train, y_train)

feat_names = ["BM25 score", "Cosine sim", "Doc length", "Term overlap"]
for name, imp in zip(feat_names, ltr_model.feature_importances_):
    print(f"  {name}: {imp:.3f}")

# --- Re-rankear ---
print("\nRe-rankeando con LTR...")
ltr_results = {}

for _, qrow in df_queries.iterrows():
    qid = qrow["query_id"]
    query_text = qrow["query"]
    candidates = bm25_results[qid][:K]
    
    feats = build_features_batch(qid, query_text, candidates, bm25_scores_dict[qid])
    ltr_scores = ltr_model.predict(feats)
    
    ranked = sorted(zip(candidates, ltr_scores), key=lambda x: x[1], reverse=True)
    ltr_results[qid] = [doc_id for doc_id, _ in ranked]

# --- Cambios de posición para query de ejemplo ---
print(f"\nQuery: {df_queries.loc[df_queries['query_id'] == example_qid, 'query'].values[0]}\n")

print(f"{'Doc ID':<12} {'BM25 Rank':>10} {'LTR Rank':>10} {'Cambio':>8}")
print("-" * 45)

ltr_rank_map = {did: i+1 for i, did in enumerate(ltr_results[example_qid][:K])}
ltr_top10 = set(ltr_results[example_qid][:10])
all_top10_ltr = bm25_top10 | ltr_top10

for doc_id in sorted(all_top10_ltr, key=lambda d: ltr_rank_map.get(d, 999)):
    br = bm25_rank_map.get(doc_id, "-")
    lr = ltr_rank_map.get(doc_id, "-")
    if isinstance(br, int) and isinstance(lr, int):
        delta = br - lr
        symbol = f"+{delta}" if delta > 0 else str(delta)
    else:
        symbol = "new" if br == "-" else "out"
    print(f"{doc_id:<12} {str(br):>10} {str(lr):>10} {symbol:>8}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10024.91it/s]


Generando embeddings del corpus...


Batches: 100%|██████████| 81/81 [02:36<00:00,  1.93s/it]


Generando embeddings de queries...


Batches: 100%|██████████| 5/5 [00:00<00:00,  6.80it/s]


Construyendo features de entrenamiento...
Dataset: 30000 pares, 4 features
  BM25 score: 0.108
  Cosine sim: 0.630
  Doc length: 0.057
  Term overlap: 0.205

Re-rankeando con LTR...

Query: Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Doc ID        BM25 Rank   LTR Rank   Cambio
---------------------------------------------
12640810              6          1       +5
86694016              8          2       +6
17934082              9          3       +6
9507605               2          4       -2
16280642             37          5      +32
35884026             28          6      +22
24294572             60          7      +53
6969753              10          8       +2
26688294              1          9       -8
25515662             98         10      +88
5270265               4         11       -7
37964706              3         12       -9
12785130              5         20 

## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [14]:
# --- Evaluar los 3 pipelines ---
results_table = {}

for name, res in [("BM25", bm25_results), ("Cross-Encoder", ce_results), ("LTR", ltr_results)]:
    ndcgs, recalls, aps = [], [], []
    
    for qid in df_queries["query_id"]:
        if qid not in qrels_dict:
            continue
        retrieved = res[qid]
        qrel = qrels_dict[qid]
        
        ndcgs.append(ndcg_at_k(retrieved, qrel, TOP_EVAL))
        recalls.append(recall_at_k(retrieved, qrel, TOP_EVAL))
        aps.append(average_precision(retrieved[:TOP_EVAL], qrel))
    
    results_table[name] = {
        "nDCG@10": np.mean(ndcgs),
        "MAP@10": np.mean(aps),
        "Recall@10": np.mean(recalls),
    }

# --- Tabla comparativa ---
df_results = pd.DataFrame(results_table).T
df_results = df_results[["nDCG@10", "MAP@10", "Recall@10"]]
print(df_results.round(4).to_string())

# --- Mejora porcentual sobre BM25 ---
print("\n--- Mejora porcentual sobre BM25 ---")
baseline = results_table["BM25"]
for name in ["Cross-Encoder", "LTR"]:
    print(f"\n{name}:")
    for metric in ["nDCG@10", "MAP@10", "Recall@10"]:
        base_val = baseline[metric]
        new_val = results_table[name][metric]
        pct = ((new_val - base_val) / base_val) * 100 if base_val > 0 else 0
        print(f"  {metric}: {base_val:.4f} → {new_val:.4f} ({pct:+.1f}%)")

               nDCG@10  MAP@10  Recall@10
BM25            0.5597  0.5147     0.6862
Cross-Encoder   0.6509  0.6134     0.7496
LTR             0.6871  0.6609     0.7563

--- Mejora porcentual sobre BM25 ---

Cross-Encoder:
  nDCG@10: 0.5597 → 0.6509 (+16.3%)
  MAP@10: 0.5147 → 0.6134 (+19.2%)
  Recall@10: 0.6862 → 0.7496 (+9.2%)

LTR:
  nDCG@10: 0.5597 → 0.6871 (+22.8%)
  MAP@10: 0.5147 → 0.6609 (+28.4%)
  Recall@10: 0.6862 → 0.7563 (+10.2%)
